## Install dependencies (spacy, flair, scispaCy models)

In [2]:
!pip install "spacy>=3.8" "thinc>=8.3" flair --quiet
!pip install --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz --quiet
!pip install --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_jnlpba_md-0.5.4.tar.gz --quiet
!pip install --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bionlp13cg_md-0.5.4.tar.gz --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 11.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.1/203.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Verify flair version

In [5]:
import en_ner_bc5cdr_md, os
pkg_dir = os.path.dirname(en_ner_bc5cdr_md.__file__)
print(pkg_dir)
!find {pkg_dir} -name "config.cfg"

/usr/local/lib/python3.12/dist-packages/en_ner_bc5cdr_md
/usr/local/lib/python3.12/dist-packages/en_ner_bc5cdr_md/en_ner_bc5cdr_md-0.5.4/config.cfg


## Locate scispaCy model config files

In [6]:
import subprocess

for pkg_name in ["en_ner_bc5cdr_md", "en_ner_jnlpba_md", "en_ner_bionlp13cg_md"]:
    mod = __import__(pkg_name)
    pkg_dir = os.path.dirname(mod.__file__)
    result = subprocess.run(["find", pkg_dir, "-name", "config.cfg"], capture_output=True, text=True)
    config_paths = result.stdout.strip().split("\n")
    for path in config_paths:
        if not path:
            continue
        print("Patching:", path)
        !sed -i 's/include_static_vectors = "True"/include_static_vectors = true/' "{path}"
        !sed -i 's/include_static_vectors = "False"/include_static_vectors = false/' "{path}"

Patching: /usr/local/lib/python3.12/dist-packages/en_ner_bc5cdr_md/en_ner_bc5cdr_md-0.5.4/config.cfg
Patching: /usr/local/lib/python3.12/dist-packages/en_ner_jnlpba_md/en_ner_jnlpba_md-0.5.4/config.cfg


/usr/local/lib/python3.12/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_ner_jnlpba_md' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Patching: /usr/local/lib/python3.12/dist-packages/en_ner_bionlp13cg_md/en_ner_bionlp13cg_md-0.5.4/config.cfg


/usr/local/lib/python3.12/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_ner_bionlp13cg_md' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


## Patch scispaCy config files for compatibility

In [8]:
import numpy
import spacy
import flair
print("numpy:", numpy.__version__)
print("spacy:", spacy.__version__)
print("flair:", flair.__version__)

nlp_test = spacy.load("en_ner_bc5cdr_md")
print("bc5cdr model loaded OK:", nlp_test.pipe_names)

numpy: 2.0.2
spacy: 3.8.14
flair: 0.15.1
bc5cdr model loaded OK: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner']


## Verify installations and load test model

In [9]:
import random
import re

import requests
import spacy
import torch
import flair
from flair.nn import Classifier
from flair.data import Sentence

## Import required libraries

In [10]:
# Must run before any model is loaded — flair reads this global at load time.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

flair.device = device

Using device: cuda


## Configure device (GPU/CPU) for model inference

In [11]:
def load_sentences_from_pmc(pmcid, max_sentences=None, seed=42):
    """
    Download an Open Access PMC article, extract its text from BioC JSON,
    and split it into sentences.
    """
    url = (
        "https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/"
        f"pmcoa.cgi/BioC_json/{pmcid}/unicode"
    )

    resp = requests.get(url)
    resp.raise_for_status()
    data = resp.json()

    collection = data[0] if isinstance(data, list) else data
    documents = collection.get("documents", [])

    text_parts = []
    for doc in documents:
        for passage in doc.get("passages", []):
            text = passage.get("text", "").strip()
            if text:
                text_parts.append(text)

    full_text = re.sub(r"\s+", " ", "\n".join(text_parts)).strip()

    sentencizer = spacy.blank("en")
    sentencizer.add_pipe("sentencizer")
    doc = sentencizer(full_text)

    sentences = [
        sent.text.strip()
        for sent in doc.sents
        if len(sent.text.strip()) > 20
    ]

    if max_sentences and len(sentences) > max_sentences:
        random.seed(seed)
        sentences = random.sample(sentences, max_sentences)

    return sentences

## Define function to load sentences from PMC article

In [12]:
FALLBACK_SENTENCES = [
    "SIRT1 activation reduces inflammation in hippocampus tissue and is "
    "associated with reduced risk of Alzheimer's disease.",
    "Metformin reduces symptoms of type 2 diabetes by activating AMPK.",
    "Overexpression of HER2 was observed in breast tissue biopsies from "
    "patients with invasive ductal carcinoma of the mammary gland.",
    "Transcription of the target mRNA was suppressed in Jurkat T cells "
    "following siRNA knockdown of the corresponding DNA sequence.",
    "The p.Val600Glu mutation in BRAF was associated with treatment "
    "resistance in melanoma patients.",
    "Resveratrol activates SIRT1 but not mTOR signaling in skeletal muscle.",
]

## Define fallback sentences for testing

In [14]:
print("Loading scispaCy bc5cdr...", flush=True)
nlp_bc5 = spacy.load("en_ner_bc5cdr_md")
print("  -> done", flush=True)

Loading scispaCy bc5cdr...
  -> done


## Load scispaCy BC5CDR model

In [15]:
print("Loading scispaCy jnlpba...", flush=True)
nlp_jnl = spacy.load("en_ner_jnlpba_md")
print("  -> done", flush=True)

Loading scispaCy jnlpba...


/usr/local/lib/python3.12/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_ner_jnlpba_md' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


  -> done


## Load scispaCy JNLPBA model

In [16]:
print("Loading scispaCy bionlp13cg...", flush=True)
nlp_bio = spacy.load("en_ner_bionlp13cg_md")
print("  -> done", flush=True)

Loading scispaCy bionlp13cg...


/usr/local/lib/python3.12/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_ner_bionlp13cg_md' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.14). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


  -> done


## Load scispaCy BioNLP13CG model

In [17]:
print("Loading HunFlair2 (GPU if available)...", flush=True)
hunflair = Classifier.load("hunflair2")  # loads onto flair.device automatically
print("  -> done", flush=True)

Loading HunFlair2 (GPU if available)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/434M [00:00<?, ?B/s]

2026-07-11 13:58:28,066 SequenceTagger predicts: Dictionary with 21 tags: O, S-Chemical, B-Chemical, E-Chemical, I-Chemical, S-Gene, B-Gene, E-Gene, I-Gene, S-Disease, B-Disease, E-Disease, I-Disease, S-Species, B-Species, E-Species, I-Species, S-CellLine, B-CellLine, E-CellLine, I-CellLine
  -> done


## Load HunFlair2 model

In [18]:
def run_scispacy(nlp, text, source_name):
    doc = nlp(text)
    return [
        {
            "source": source_name,
            "text": ent.text,
            "label": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char,
        }
        for ent in doc.ents
    ]


def run_hunflair2(hunflair, text):
    sentence = Sentence(text)
    hunflair.predict(sentence)
    results = []
    for ent in sentence.get_spans("ner"):
        results.append(
            {
                "source": "hunflair2",
                "text": ent.text,
                "label": ent.get_label("ner").value,
                "start": ent.start_position,
                "end": ent.end_position,
                "confidence": round(ent.get_label("ner").score, 3),
            }
        )
    return results

## Define helper functions for NER inference

In [23]:
def print_comparison(sentence_idx, text, all_results):
    print("\n" + "=" * 100)
    print(f"SENTENCE {sentence_idx + 1}: {text}")
    print("=" * 100)

    by_source = {}
    for r in all_results:
        by_source.setdefault(r["source"], []).append(r)

    for source in ["scispacy_bc5cdr", "scispacy_jnlpba", "scispacy_bionlp13cg", "hunflair2"]:
        ents = by_source.get(source, [])
        print(f"\n  [{source}]  ({len(ents)} entities)")
        if not ents:
            print("      (none found)")
        for e in sorted(ents, key=lambda x: x["start"]):
            conf = f"  conf={e['confidence']}" if "confidence" in e else ""
            print(f"      '{e['text']}'  ->  {e['label']}{conf}  [{e['start']}:{e['end']}]")

    # Flag spans only one model caught (useful for spotting real coverage gaps)
    all_spans = [(e["start"], e["end"], e["source"]) for e in all_results]
    span_sources = {}
    for start, end, source in all_spans:
        span_sources.setdefault((start, end), set()).add(source)

    unique_catches = {k: v for k, v in span_sources.items() if len(v) == 1}
    if unique_catches:
        print("\n  --- Caught by only ONE model (possible coverage gap elsewhere) ---")
        for (start, end), sources in unique_catches.items():
            matching = [e for e in all_results if e["start"] == start and e["end"] == end]
            if matching:
                e = matching[0]
                print(f"      '{e['text']}' ({e['label']}) — only {list(sources)[0]}")

## Define comparison printing function

In [24]:
PMCID = "PMC7390530"   # Change to any Open Access PMC ID
MAX_SENTENCES = 100
SEED = 42

sentences = load_sentences_from_pmc(PMCID, max_sentences=MAX_SENTENCES, seed=SEED)
print(f"Loaded {len(sentences)} sentences from {PMCID}")

Loaded 100 sentences from PMC7390530


## Load sentences from PMC article

In [25]:
for idx, text in enumerate(sentences):
    results = []
    results += run_scispacy(nlp_bc5, text, "scispacy_bc5cdr")
    results += run_scispacy(nlp_jnl, text, "scispacy_jnlpba")
    results += run_scispacy(nlp_bio, text, "scispacy_bionlp13cg")
    results += run_hunflair2(hunflair, text)

    print_comparison(idx, text, results)

print("\n" + "=" * 100)
print("Done.")
print("=" * 100)


SENTENCE 1: Recent studies have shown that sirt6 also has a deacylase activity and interacts physically with some non-histone proteins - not only through deacylation, but also through direct physical interaction (PIA), inhibition of their binding at the target gene promoters (IATGP) and destabilization of their binding at the target gene promoters (DATGP).

  [scispacy_bc5cdr]  (0 entities)
      (none found)

  [scispacy_jnlpba]  (4 entities)
      'deacylase'  ->  PROTEIN  [48:57]
      'non-histone proteins'  ->  PROTEIN  [102:122]
      'target gene promoters'  ->  DNA  [242:263]
      'target gene promoters'  ->  DNA  [316:337]

  [scispacy_bionlp13cg]  (3 entities)
      'sirt6'  ->  CELL  [31:36]
      'PIA'  ->  SIMPLE_CHEMICAL  [201:204]
      'DATGP'  ->  CELLULAR_COMPONENT  [339:344]

  [hunflair2]  (1 entities)
      'sirt6'  ->  Gene  conf=1.0  [31:36]

  --- Caught by only ONE model (possible coverage gap elsewhere) ---
      'deacylase' (PROTEIN) — only scispacy_jnlpba
